In [ ]:
# CELL 1 — CONFIGURATION, SOURCE DISCOVERY & SQL ENGINE
from pathlib import Path
import sys, subprocess, importlib.util, re, sqlite3, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
OUT = ROOT / 'Project10_EV_Outputs'
OUT.mkdir(parents=True, exist_ok=True)
# If discovery chooses the wrong file, specify the absolute path here:
DATA_FILE = None
INFRA_FILE = None  # optional CSV with state-level charging-station counts

def list_csvs():
    candidates = [p for p in ROOT.glob('*.csv') if p.is_file()]
    if not candidates and (ROOT/'drive').exists():
        print('Google Drive file? Set DATA_FILE to its full path after mounting Drive.')
    return [p for p in candidates if not p.name.startswith(('us_ev_', 'p10_'))]

if DATA_FILE is None:
    candidates = list_csvs()
    assert candidates, 'Upload the vehicle registration CSV to /content or set DATA_FILE to its path in Cell 1.'
    ranked = sorted(candidates, key=lambda p: (int(any(w in p.stem.lower() for w in ['vehicle','fuel','registration','state','ev'])),p.stat().st_size), reverse=True)
    DATA_FILE = ranked[0]
DATA_FILE = Path(DATA_FILE)
assert DATA_FILE.exists(), f'Missing data file: {DATA_FILE}'
if INFRA_FILE is not None: INFRA_FILE = Path(INFRA_FILE)
DB_PATH = OUT / 'alternative_fuel_vehicles.sqlite'
con = sqlite3.connect(str(DB_PATH))
print('Registration dataset:', DATA_FILE.name)
print('Other CSVs detected:', [x.name for x in list_csvs() if x != DATA_FILE])
print('SQL engine: SQLite | Outputs:', OUT)


In [ ]:
# CELL 2 — INSPECT THE REAL SCHEMA, CLEAN STATE/FUEL LABELS, AUDIT QUALITY
# Accepts WIDE (State, Electric, PHEV, HEV, Gasoline...) or LONG (State, Fuel Type, Vehicles).
raw = pd.read_csv(DATA_FILE, low_memory=False)
assert len(raw)>0, 'Source CSV contains no records.'
print('Source shape:',raw.shape);print('Source columns:',list(raw.columns));display(raw.head(4))
def norm(x): return re.sub(r'[^a-z0-9]+','_',str(x).strip().lower()).strip('_')
cols={norm(x):x for x in raw.columns}
def locate(*names):
    for name in names:
        if norm(name) in cols:return cols[norm(name)]
    return None
state_col=locate('state','state_name','state abbreviation','state_abbr','jurisdiction')
assert state_col, 'Cannot identify state column. Paste the printed column names for a one-cell correction.'
year_col=locate('year','registration_year','model_year','calendar_year','data_year')
fuel_col=locate('fuel_type','fuel','fuel_category','vehicle_fuel','technology','powertrain')
count_col=locate('vehicle_count','vehicles','count','registrations','number_of_vehicles','registered_vehicles','total_vehicles')
STATE_ABBR=dict(zip('AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC'.split(),
'Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|West Virginia|Wisconsin|Wyoming|District of Columbia'.split('|')))
state_lookup={norm(v):v for v in STATE_ABBR.values()};state_lookup.update({norm(k):v for k,v in STATE_ABBR.items()})
def state_name(v):
    if pd.isna(v):return None
    t=norm(v);return state_lookup.get(t, str(v).strip().title() if t else None)

def fuel_name(x):
    t=norm(x)
    if t in ('ev','bev','battery_electric','battery_electric_vehicle','all_electric','electric','electric_vehicle','electric_vehicles','battery_electric_vehicles'):return 'EV'
    if t in ('phev','plug_in_hybrid','plug_in_hybrid_electric','plug_in_hybrid_electric_vehicle','plug_in_hybrid_vehicles'):return 'PHEV'
    if t in ('hev','hybrid','hybrid_electric','hybrid_electric_vehicle','hybrid_electric_vehicles','gasoline_hybrid'):return 'HEV'
    if t in ('gas','gasoline','petrol','gasoline_vehicles','gasoline_vehicle'):return 'GASOLINE'
    if 'biodiesel' in t:return 'BIODIESEL'
    if 'ethanol' in t or t in ('e85','flex_fuel','flexible_fuel'):return 'ETHANOL/FLEX FUEL'
    if 'hydrogen' in t or t in ('fcev','fuel_cell','fuel_cell_electric'):return 'HYDROGEN'
    if 'diesel' in t:return 'DIESEL'
    if 'natural_gas' in t or t in ('cng','lng'):return 'NATURAL GAS'
    if 'propane' in t or 'lpg' in t:return 'PROPANE'
    return str(x).strip().upper() if not pd.isna(x) and str(x).strip() else 'UNKNOWN'

def numeric(s): return pd.to_numeric(s.astype(str).str.replace(',','',regex=False).str.replace('$','',regex=False).str.strip(),errors='coerce')
raw['__state']=raw[state_col].map(state_name)
raw['__year']=numeric(raw[year_col]).astype('Int64') if year_col else pd.Series(pd.array([pd.NA]*len(raw),dtype='Int64'))
raw['__year']=raw['__year'].where(raw['__year'].between(1990,2100))
if fuel_col and count_col:
    raw['__fuel']=raw[fuel_col].map(fuel_name)
    raw['__count']=numeric(raw[count_col])
    source_long=raw[['__state','__year','__fuel','__count']].rename(columns={'__state':'state','__year':'year','__fuel':'fuel','__count':'vehicles'})
    schema='LONG'
else:
    reserved={x for x in (state_col,year_col) if x is not None}
    numerics=[]
    for col in raw.columns:
        if col in reserved or str(col).startswith('__'):continue
        parsed=numeric(raw[col]);rate=parsed.notna().mean()
        if rate>=0.7 and (any(w in norm(col) for w in ('ev','electric','phev','hev','hybrid','gas','diesel','ethanol','hydrogen','biodiesel','cng','natural','propane','fuel','vehicle','total')) or rate>=0.98):numerics.append(col)
    assert numerics, 'Could not identify numeric fuel columns. Send the column names for a Cell 2 adjustment.'
    # TOTAL is a denominator reference, not a fuel category. Keep separate for checking.
    total_cols=[x for x in numerics if norm(x) in ('total','total_vehicles','all_vehicles','total_registrations','all_registered_vehicles')]
    fuel_columns=[x for x in numerics if x not in total_cols]
    assert fuel_columns, 'No individual fuel-count columns found.'
    frames=[]
    for col in fuel_columns:
        t=pd.DataFrame({'state':raw['__state'],'year':raw['__year'],'fuel':fuel_name(col),'vehicles':numeric(raw[col])});frames.append(t)
    source_long=pd.concat(frames,ignore_index=True)
    schema='WIDE'
# Do not silently convert missing counts or negative counts to zero.
source_long['vehicles']=pd.to_numeric(source_long['vehicles'],errors='coerce')
quality={'source_rows':len(raw),'source_columns':len(raw.columns)-(3 if schema=='LONG' else 2),'schema':schema,'missing_state_rows':int(raw['__state'].isna().sum()),'missing_vehicle_counts':int(source_long.vehicles.isna().sum()),'negative_vehicle_counts':int((source_long.vehicles<0).sum()),'fractional_vehicle_counts':int(((source_long.vehicles%1!=0)&source_long.vehicles.notna()).sum()),'unknown_fuel_rows':int((source_long.fuel=='UNKNOWN').sum()),'source_has_year':bool(year_col)}
valid=source_long.loc[source_long.state.notna() & source_long.vehicles.notna() & (source_long.vehicles>=0) & (source_long.vehicles%1==0)].copy()
valid['vehicles']=valid['vehicles'].astype('int64')
# Snapshot policy: if years are provided, use the latest year only for adoption rankings.
years=sorted(int(v) for v in valid.year.dropna().unique());latest_year=years[-1] if years else None
if latest_year is not None: snapshot=valid.loc[valid.year==latest_year].copy()
else: snapshot=valid.copy()
quality.update({'latest_snapshot_year':latest_year,'valid_long_records':len(valid),'snapshot_long_records':len(snapshot),'distinct_states_snapshot':snapshot.state.nunique(),'excluded_missing_or_invalid_records':len(source_long)-len(valid)})
quality_df=pd.DataFrame([quality]);quality_df.to_csv(OUT/'us_ev_data_quality_audit.csv',index=False)
valid.to_sql('registration_records',con,if_exists='replace',index=False)
snapshot.to_sql('snapshot_records',con,if_exists='replace',index=False)
print('Detected shape:',schema,'| Latest year:',latest_year,'| Valid records:',len(valid));display(quality_df.T)
print('Fuel labels:', sorted(valid.fuel.unique()))


In [ ]:
# CELL 3 — VALIDATE DENOMINATORS, BUILD STATE/FUEL SQL MARKET SHARES
# Critical: do not sum a supplied TOTAL column alongside individual fuel types.
state_fuel=pd.read_sql_query('SELECT state, fuel, SUM(vehicles) AS vehicle_count FROM snapshot_records GROUP BY state,fuel ORDER BY state,fuel',con)
assert not state_fuel.empty, 'No state/fuel observations after cleaning.'
# If long format has explicit TOTAL rows, strip them from fuel categories.
total_names={'TOTAL','TOTAL VEHICLES','ALL VEHICLES','ALL FUELS','TOTAL REGISTRATIONS'}
explicit_totals=state_fuel[state_fuel.fuel.isin(total_names)].groupby('state')['vehicle_count'].sum()
state_fuel=state_fuel[~state_fuel.fuel.isin(total_names)].copy()
# Other uncategorized fuels remain in denominator to avoid inflating EV market share.
state_total=state_fuel.groupby('state')['vehicle_count'].sum()
assert (state_total>0).any(), 'All state totals are zero.'
if len(explicit_totals):
    discrepancy=pd.DataFrame({'calculated_category_total':state_total,'source_total':explicit_totals}).dropna()
    discrepancy['difference']=discrepancy.calculated_category_total-discrepancy.source_total
    discrepancy.to_csv(OUT/'us_ev_total_denominator_discrepancy_audit.csv')
    print('Source total vs summed fuel categories (inspect discrepancies):');display(discrepancy.head())
state_fuel['total_registered_fuel_categories']=state_fuel.state.map(state_total)
state_fuel['market_share_pct']=np.where(state_fuel.total_registered_fuel_categories>0,100*state_fuel.vehicle_count/state_fuel.total_registered_fuel_categories,np.nan)
state_fuel.to_csv(OUT/'us_ev_state_fuel_market_shares.csv',index=False)
key=['EV','PHEV','HEV','GASOLINE','DIESEL','BIODIESEL','ETHANOL/FLEX FUEL','HYDROGEN']
pivot=state_fuel.pivot_table(index='state',columns='fuel',values='vehicle_count',aggfunc='sum',fill_value=0)
for x in key:
    if x not in pivot.columns:pivot[x]=0
summary=pivot[key].copy();summary['total_vehicles']=state_total
for x in key:summary[x.lower().replace('/','_').replace(' ','_')+'_share_pct']=np.where(summary.total_vehicles>0,100*summary[x]/summary.total_vehicles,np.nan)
summary['ev_adoption_pct']=summary['ev_share_pct'];summary['plugin_ev_share_pct']=summary['ev_share_pct']+summary['phev_share_pct']
summary=summary.reset_index().sort_values('ev_adoption_pct',ascending=False)
summary.to_csv(OUT/'us_ev_state_adoption_rankings.csv',index=False)
# Checks: every state has shares summing to 100% across ALL encountered non-total fuel categories.
recon=state_fuel.groupby('state').market_share_pct.sum();assert np.allclose(recon.loc[state_total[state_total>0].index].values,100,atol=.0001)
assert not summary.empty
print('MARKET SHARE VALIDATION: PASS | States:',len(summary));display(summary[['state','total_vehicles','EV','PHEV','HEV','GASOLINE','ev_adoption_pct']].head(10))


In [ ]:
# CELL 4 — SQL RANKINGS, CALIFORNIA BENCHMARK, ALTERNATIVE FUEL PRESENCE
state_fuel[['state','fuel','vehicle_count','market_share_pct']].to_sql('state_fuel_market_shares',con,if_exists='replace',index=False)
summary.to_sql('state_adoption',con,if_exists='replace',index=False)
top5=pd.read_sql_query('SELECT state, EV AS ev_vehicles, total_vehicles, ROUND(ev_adoption_pct,3) ev_adoption_pct FROM state_adoption WHERE total_vehicles>0 ORDER BY ev_adoption_pct DESC, EV DESC LIMIT 5',con)
bottom5=pd.read_sql_query('SELECT state, EV AS ev_vehicles, total_vehicles, ROUND(ev_adoption_pct,3) ev_adoption_pct FROM state_adoption WHERE total_vehicles>0 ORDER BY ev_adoption_pct ASC, total_vehicles DESC LIMIT 5',con)
large_states=pd.read_sql_query("SELECT state,total_vehicles,EV,PHEV,HEV,GASOLINE,ROUND(ev_adoption_pct,3) ev_adoption_pct FROM state_adoption WHERE state IN ('California','Texas','Florida','New York') ORDER BY ev_adoption_pct DESC",con)
fuel_overview=pd.read_sql_query('SELECT fuel,SUM(vehicle_count) AS registered_vehicles, ROUND(100.0*SUM(vehicle_count)/(SELECT SUM(vehicle_count) FROM state_fuel_market_shares),5) AS national_share_pct,COUNT(DISTINCT CASE WHEN vehicle_count>0 THEN state END) states_reporting_positive FROM state_fuel_market_shares GROUP BY fuel ORDER BY registered_vehicles DESC',con)
for name,frame in [('top5_ev_states',top5),('bottom5_ev_states',bottom5),('large_state_comparison',large_states),('national_fuel_mix',fuel_overview)]:frame.to_csv(OUT/f'us_ev_{name}.csv',index=False)
print('TOP FIVE EV SHARE');display(top5)
print('BOTTOM FIVE EV SHARE');display(bottom5)
print('CALIFORNIA VS LARGE STATES');display(large_states)
print('ALL FUELS—COUNTS & SHARE');display(fuel_overview)
# Small positive totals are PRESENCE not evidence of commercial significance; inspect per-fuel shares.


In [ ]:
# CELL 5 — TREND COVERAGE, INFRASTRUCTURE GAP EVIDENCE, THREE POLICY CANDIDATES
if len(years)>=2:
    trends=pd.read_sql_query('SELECT year,state,fuel,SUM(vehicles) vehicles FROM registration_records WHERE year IS NOT NULL GROUP BY year,state,fuel',con)
    totals=trends.loc[~trends.fuel.isin({'TOTAL','TOTAL VEHICLES','ALL VEHICLES','ALL FUELS','TOTAL REGISTRATIONS'})].groupby(['year','state']).vehicles.sum().rename('denominator')
    evs=trends[trends.fuel=='EV'].set_index(['year','state']).vehicles
    trend=(pd.concat([totals,evs.rename('ev')],axis=1).fillna({'ev':0}).reset_index())
    trend['ev_adoption_pct']=np.where(trend.denominator>0,100*trend.ev/trend.denominator,np.nan)
    trend.to_csv(OUT/'us_ev_state_yearly_adoption.csv',index=False)
    print('Time trends available:',len(years),'years:',years)
else:
    trend=pd.DataFrame(columns=['year','state','denominator','ev','ev_adoption_pct']);trend.to_csv(OUT/'us_ev_state_yearly_adoption.csv',index=False)
    print('Only one/no year detected: cannot calculate adoption growth; latest snapshot only.')
# Optional infrastructure coverage. Required schema: a state field + charging station count.
infra=pd.DataFrame(columns=['state','charging_stations'])
if INFRA_FILE is not None:
    assert INFRA_FILE.exists(),f'INFRA_FILE not found: {INFRA_FILE}'
    tmp=pd.read_csv(INFRA_FILE,low_memory=False);cmap={norm(c):c for c in tmp.columns}
    sc=next((cmap[x] for x in ['state','state_name','state_abbr'] if x in cmap),None)
    ic=next((cmap[x] for x in ['charging_stations','station_count','ev_charging_stations','charging_ports'] if x in cmap),None)
    assert sc and ic,'Infrastructure CSV must have state and charging_stations (or charging_ports) columns.'
    infra=pd.DataFrame({'state':tmp[sc].map(state_name),'charging_stations':numeric(tmp[ic])}).groupby('state',as_index=False).charging_stations.sum()
    infra.to_csv(OUT/'us_ev_infrastructure_by_state.csv',index=False)
# Select 3 candidate states using combination of EV count + market share (screening ONLY).
# Don't claim they lack chargers unless an infrastructure table is provided.
priority=summary.loc[summary.total_vehicles>0,['state','EV','total_vehicles','ev_adoption_pct']].copy()
priority['ev_volume_rank_pct']=priority.EV.rank(pct=True)
priority['ev_share_rank_pct']=priority.ev_adoption_pct.rank(pct=True)
priority['screening_score']=((priority.ev_volume_rank_pct+priority.ev_share_rank_pct)/2).round(4)
if not infra.empty:
    priority=priority.merge(infra,on='state',how='left')
    priority['evs_per_station']=np.where(priority.charging_stations>0,priority.EV/priority.charging_stations,np.nan)
    # A station and a port are not interchangeable; validate units before treating this as capacity.
    priority['priority_basis']='EV volume/share screen; cross-check charger coverage and station capacity'
else:
    priority['priority_basis']='Provisional EV volume/share screen ONLY; charging infrastructure data unavailable'
priority=priority.sort_values(['screening_score','EV'],ascending=False)
priority.to_csv(OUT/'us_ev_infrastructure_priority_screen.csv',index=False)
priority3=priority.head(3).copy();priority3.to_csv(OUT/'us_ev_three_priority_state_candidates.csv',index=False)
print('THREE PROVISIONAL POLICY CANDIDATES');display(priority3)
print('IMPORTANT: Without charger counts/location/capacity, these are candidate states—not verified infrastructure gaps.')


In [ ]:
# CELL 6 — EXECUTIVE VISUALIZATION, FINAL RECONCILIATION
fig,ax=plt.subplots(2,2,figsize=(15,10))
top=summary.nlargest(12,'ev_adoption_pct').iloc[::-1]
ax[0,0].barh(top.state,top.ev_adoption_pct);ax[0,0].set_title('Top states: EV share of registered vehicles');ax[0,0].set_xlabel('EV adoption (%)')
fuels=fuel_overview.head(9).iloc[::-1];ax[0,1].barh(fuels.fuel,fuels.registered_vehicles);ax[0,1].set_title('Registered vehicles by fuel type');ax[0,1].set_xlabel('Vehicle counts')
ax[1,0].bar(large_states.state,large_states.ev_adoption_pct);ax[1,0].set_title('California versus selected large states');ax[1,0].set_ylabel('EV share (%)');ax[1,0].tick_params(axis='x',rotation=20)
if not trend.empty:
    for state in ['California','Texas','Florida','New York']:
        d=trend.loc[trend.state==state].sort_values('year')
        if len(d):ax[1,1].plot(d.year,d.ev_adoption_pct,marker='o',label=state)
    ax[1,1].legend(fontsize=8);ax[1,1].set_title('EV adoption trend — states with multiple years');ax[1,1].set_ylabel('EV share (%)')
else:
    x=summary.nlargest(8,'EV').iloc[::-1];ax[1,1].barh(x.state,x.EV);ax[1,1].set_title('Largest EV fleets (single snapshot)');ax[1,1].set_xlabel('Registered EVs')
for a in ax.flat:a.grid(alpha=.16);a.set_axisbelow(True)
fig.suptitle('U.S. Alternative Fuel Vehicle Adoption | Descriptive SQL Dashboard',fontsize=15)
fig.tight_layout(rect=(0,0,1,.96));fig.savefig(OUT/'us_ev_adoption_executive_dashboard.png',dpi=155,bbox_inches='tight');plt.show()
check=state_fuel.groupby('state').vehicle_count.sum();assert all(check.loc[summary.state].values==summary.total_vehicles.values)
assert int(fuel_overview.registered_vehicles.sum())==int(state_fuel.vehicle_count.sum())
print('VALIDATION: PASS | state totals reconcile | dashboard saved')


In [ ]:
# CELL 7 — DATA-GROUNDED README, 3 PRIORITY STATES, SEVEN ACTIONS & VERIFIED ZIP
from zipfile import ZipFile,ZIP_DEFLATED

def md(frame, cols=None,limit=10):
    if frame.empty:return 'No records available.'
    t=frame.loc[:,cols] if cols else frame;t=t.head(limit).copy()
    for x in t.select_dtypes(include='number').columns:t[x]=t[x].round(3)
    out=['| '+' | '.join(t.columns)+' |','| '+' | '.join(['---']*len(t.columns))+' |']
    for values in t.itertuples(index=False,name=None):out.append('| '+' | '.join('—' if pd.isna(x) else str(x).replace('|','/') for x in values)+' |')
    return '\n'.join(out)

priority_names=', '.join(priority3.state.tolist()) if len(priority3) else 'None available'
period=(f'{years[0]}–{years[-1]}' if len(years)>=2 else f'{latest_year} snapshot' if latest_year else 'undated snapshot')
infra_note=('An optional infrastructure file was provided; verify whether the supplied values count stations or charging ports and assess actual spatial coverage.' if not infra.empty else 'No verified public charging-station coverage/capacity data was supplied. Infrastructure gaps cannot be measured or asserted from registrations alone.')
readme=f'''# U.S. Alternative Fuel Vehicles — State Adoption & Infrastructure Screening

![Executive dashboard](us_ev_adoption_executive_dashboard.png)

## Policy objective
Compare EV/PHEV/HEV/gasoline registration shares and other fuel categories; screen states for follow-up charging-network investment analysis without inventing missing infrastructure or time trends.

## Source and scope
- Input: `{DATA_FILE.name}`; format: **{schema}**; period: **{period}**.
- Source rows: **{len(raw):,}**; valid fuel-state records after cleaning: **{len(valid):,}**.
- **{len(summary)}** state/jurisdiction labels in the snapshot. Fuel shares use the sum of mutually exclusive observed fuel-category counts, **not** population or vehicle-miles driven.
- If source fuel categories overlap (for example 'flex-fuel' labels might overlap gasoline), percentages cannot be interpreted as a single mutually exclusive registered-fleet mix until source definitions are checked.
- Missing/invalid counts are excluded, not assumed zero. Zero reported registrations are valid observations but may also reflect incomplete reporting.
- Latest-year snapshot is used for rankings; multi-year trends are shown only if distinct years exist.

## Executive state comparisons
{md(summary[['state','total_vehicles','EV','PHEV','HEV','GASOLINE','ev_adoption_pct']])}

## Top five states by EV market share
{md(top5)}

## Bottom five states by EV market share
{md(bottom5)}

## California compared with large states
{md(large_states)}

## National fuel category distribution
{md(fuel_overview,['fuel','registered_vehicles','national_share_pct','states_reporting_positive'],20)}

## Adoption trends
{('Multiple years were supplied; see `us_ev_state_yearly_adoption.csv` for time-series rates. Compare consistent reporting practices between years.' if len(years)>=2 else 'Only one registration snapshot was supplied. EV adoption growth and accelerating/decelerating adoption **cannot be determined** from this file.')}

## Three provisional infrastructure investment candidates
**{priority_names}** — selected using a transparent combined EV fleet-count and adoption-share screening score, **not** a documented charger shortfall.
{md(priority3[['state','EV','total_vehicles','ev_adoption_pct','screening_score']])}

{infra_note}

## Data quality and methodology
1. Identify wide or long registration schema and standardize state abbreviations/names plus common fuel labels.
2. Audit missing, negative and fractional registration counts; retain original reported labels for unknown fuel categories.
3. Use the most recent valid year for current snapshot rankings; don't double-count source TOTAL categories.
4. Aggregate fuel counts per state with SQLite; compute EV, PHEV, HEV, gasoline percentages using observed totals.
5. Reconcile national counts and state shares; flag any source-total discrepancies.
6. Review fuel classifications: biodiesel blends, ethanol/flex-fuel compatibility and hydrogen registrations may not be comparable or disjoint without the dataset dictionary.
7. Export audited CSVs, static Python dashboard, recommendations, and a validated ZIP.

## Files / reproduction
Open `US_Alternative_Fuel_Vehicle_Adoption_SQL_Analysis.ipynb` in Google Colab, upload the original CSV to `/content`, optionally set `DATA_FILE` and `INFRA_FILE`, run all seven cells, and check `VALIDATION: PASS` and `PACKAGE READY`. Results save to `/content/Project10_EV_Outputs`. The notebook is downloaded separately from the ZIP. A local SQLite database is intentionally excluded from GitHub packaging.

## Technology and limitations
Python, pandas, SQLite SQL, Matplotlib and Excel-compatible CSVs. The dashboard is static Python; **no Tableau workbook is claimed**. Without charging-station/port data, electricity grid capacity, utilization, traffic, per-capita registrations, and regulatory context, prioritization remains preliminary. This dataset cannot measure actual adoption *growth* without multiple comparable years.

## Repository
`US-Alternative-Fuel-Vehicle-Adoption-Analytics` | https://github.com/new
'''
(OUT/'README.md').write_text(readme,encoding='utf-8')
recommend=f'''# U.S. Alternative Fuel Vehicles — Transportation Policy Recommendations

## Interpretation standard
Vehicle registrations measure recorded fleet stock—not new purchases, charging access, infrastructure shortage, charging utilization, or causal policy outcomes.

## 1. Screen three states for infrastructure feasibility (HIGH)
**Candidates:** {priority_names}. **Evidence:** latest EV market-share and absolute EV-stock screening in `us_ev_three_priority_state_candidates.csv`. **Action:** collect and validate charger locations, number of ports, uptime, utilization, housing types, corridor traffic and underserved community access before siting new investments. **KPI:** operational public ports per 1,000 EVs, wait time and access within a travel-time threshold. **Caveat:** no charger gap is established from registrations alone.

## 2. Prioritize verified underserved charging areas (CRITICAL)
**Action:** overlay geocoded charger locations, household income, apartment occupancy and trips with EV registration concentration. **KPI:** geographic coverage and reliable fast-charge availability. Avoid assuming state-level registration share reveals local access gaps.

## 3. Plan separately for EVs and PHEVs (HIGH)
**Action:** distinguish battery-electric fast-charging needs from plug-in hybrid mixed-fuel behavior. **KPI:** charger demand per plug-in vehicle by connection type, measured utilization.

## 4. Benchmark lagging states with comparable denominators (MEDIUM)
**Evidence:** five lowest shares appear in `us_ev_bottom5_ev_states.csv`. **Action:** check data completeness, income, travel needs and vehicle mix before considering policy incentives. **KPI:** annual change in EV share after matching comparable registration periods.

## 5. Evaluate hybrid and gasoline transition pathways (HIGH)
**Action:** use `us_ev_state_fuel_market_shares.csv` to identify high conventional-fleet states and develop grid-capacity and dealer-readiness scenarios. **KPI:** EV registrations as a share of replacements and grid upgrades completed.

## 6. Classify non-electric alternative fuels correctly (MEDIUM)
**Action:** separately track biodiesel blends, ethanol flex-fuel capability and hydrogen fuel-cell registrations using source definitions. **KPI:** verified distinct registrations, fueling availability, and fleet use cases; avoid treating overlapping fuel labels as disjoint.

## 7. Publish a multi-year data and infrastructure scorecard (CRITICAL)
**Action:** collect comparable years, source denominators, public charging stations/ports, and data refresh dates. **KPI:** missingness, reconciliation error, adoption share by year and charger uptime; publish assumptions and exclusions.

## Decision
Treat the three selected states as **follow-up study candidates**, not evidence of a verified charger shortage. Confirm with charging infrastructure and traffic data before authorizing capital expenditures.
'''
(OUT/'RECOMMENDATIONS.md').write_text(recommend,encoding='utf-8')
(OUT/'requirements.txt').write_text('pandas>=2.0,<4\nnumpy>=1.24,<3\nmatplotlib>=3.7,<4\n',encoding='utf-8')
(OUT/'.gitignore').write_text('*.sqlite\n*.db\n*.wal\n.ipynb_checkpoints/\n__pycache__/\n.DS_Store\n',encoding='utf-8')
package=ROOT/'Project10_EV_Adoption_GitHub_Package.zip'
pack=[f for f in OUT.iterdir() if f.is_file() and f.suffix.lower() in ('.csv','.png','.md','.txt')]
pack.append(OUT/'.gitignore')
required=['README.md','RECOMMENDATIONS.md','us_ev_state_adoption_rankings.csv','us_ev_state_fuel_market_shares.csv','us_ev_adoption_executive_dashboard.png']
assert all((OUT/name).exists() for name in required)
with ZipFile(package,'w',ZIP_DEFLATED) as z:
    for f in pack:z.write(f,arcname=f.name)
with ZipFile(package) as z:
    assert z.testzip() is None
    assert all(x in z.namelist() for x in required)
print('VALIDATION: PASS | ZIP INTEGRITY: PASS | Files:',len(pack))
print('PACKAGE READY:',package)
print('Download .ipynb separately from Colab; do not upload SQLite database or redundant ZIP to GitHub.')
if Path('/content').exists():
    from google.colab import files
    files.download(str(package))
